# 02 — Chunking Demo

Full pipeline: **PDF → extract blocks → classify → chunk**

This notebook demonstrates Layers 1 → 2 → 4 of the HSC-Edu pipeline,
ending with semantic chunks ready for embedding.

In [1]:
from pathlib import Path

from hsc_edu.core.extraction import extract_document
from hsc_edu.core.classification import classify_blocks
from hsc_edu.core.chunking import chunk_blocks, count_tokens
from hsc_edu.core.models import BlockType

DATA_DIR = Path("../data")
SAMPLE_PDF = DATA_DIR / "Java.pdf"

print(f"PDF: {SAMPLE_PDF.name}  ({SAMPLE_PDF.stat().st_size / 1024 / 1024:.1f} MB)")

PDF: Java.pdf  (6.3 MB)


## 1. Extract blocks (Layer 1)

In [2]:
blocks = extract_document(SAMPLE_PDF, doc_id="java-demo")
print(f"Extracted {len(blocks)} blocks from '{SAMPLE_PDF.name}'")

Extracted 2121 blocks from 'Java.pdf'


## 2. Classify blocks (Layer 2)

In [3]:
classified = classify_blocks(blocks)

from collections import Counter

type_counts = Counter(b.block_type for b in classified)
print(f"Classified {len(classified)} blocks:")
for bt, cnt in type_counts.most_common():
    print(f"  {bt:<20s} {cnt:>5d}")

Classified 2121 blocks:
  paragraph             1884
  heading                223
  exercise                13
  note                     1


In [4]:
headings = [b for b in classified if b.block_type == BlockType.HEADING]
print(f"Headings found: {len(headings)}\n")

for h in headings[:15]:
    indent = "  " * ((h.heading_level or 1) - 1)
    text = h.raw_text.split("\n")[0][:80]
    print(f"  p.{h.page:>3d}  lv{h.heading_level}  {indent}{text}")

Headings found: 223

  p.  0  lv3      2.4.3. Biểu thức điều kiện trong các cấu trúc điều khiển 43
  p.  1  lv2    5.1. PHƯƠNG THỨC VÀ TRẠNG THÁI ĐỐI TƯỢNG70
  p.  1  lv2    5.4. ĐÓNG GÓI VÀ CÁC PHƯƠNG THỨC TRUY NHẬP 
  p.  1  lv2    7.8. GỌI PHIÊN BẢN PHƯƠNG THỨC CỦA LỚP CHA114
  p.  2  lv2    8.6. ĐỔI KIỂU – KHI ĐỐI TƯỢNG MẤT HÀNH VI CỦA MÌNH 
  p.  2  lv1  Chương 10. THÀNH VIÊN LỚP VÀ THÀNH VIÊN THỰC THỂ 
  p.  2  lv3      11.2.3. Khối finally – những việc dù thế nào cũng phải làm 
  p.  3  lv2    11.5. NGOẠI LỆ ĐƯỢC KIỂM TRA VÀ KHÔNG ĐƯỢC KIỂM TRA 
  p.  3  lv2    11.7. NGOẠI LỆ VÀ CÁC PHƯƠNG THỨC CÀI ĐÈ . 191
  p.  3  lv1  Chương 12. CHUỖI HÓA ĐỐI TƯỢNG VÀ VÀO RA FILE 
  p.  3  lv1  Chương 13. LẬP TRÌNH TỔNG QUÁT VÀ CÁC LỚP COLLECTION 
  p.  3  lv2    13.3. CÁC CẤU TRÚC DỮ LIỆU TỔNG QUÁT TRONG JAVA API 220
  p.  3  lv2    13.6. KÍ TỰ ĐẠI DIỆN TRONG KHAI BÁO THAM SỐ KIỂU 228
  p. 11  lv2    1.1. KHÁI NIỆM CƠ BẢN
  p. 12  lv2    1.2. ĐỐI TƯỢNG VÀ LỚP


## 3. Chunk (Layer 4)

In [5]:
chunks = chunk_blocks(classified)

print(f"Total chunks: {len(chunks)}")
print(f"Token range : {min(c.token_count for c in chunks)} – {max(c.token_count for c in chunks)}")
print(f"Avg tokens  : {sum(c.token_count for c in chunks) / len(chunks):.0f}")

Total chunks: 332
Token range : 12 – 1064
Avg tokens  : 535


### 3.1 Token distribution

In [6]:
buckets = {"0-64": 0, "64-256": 0, "256-512": 0, "512-1024": 0, ">1024": 0}
for c in chunks:
    t = c.token_count
    if t <= 64:
        buckets["0-64"] += 1
    elif t <= 256:
        buckets["64-256"] += 1
    elif t <= 512:
        buckets["256-512"] += 1
    elif t <= 1024:
        buckets["512-1024"] += 1
    else:
        buckets[">1024"] += 1

print("Token distribution:")
for label, cnt in buckets.items():
    bar = "█" * cnt
    print(f"  {label:>10s}: {cnt:>4d}  {bar}")

Token distribution:
        0-64:   88  ████████████████████████████████████████████████████████████████████████████████████████
      64-256:   26  ██████████████████████████
     256-512:   40  ████████████████████████████████████████
    512-1024:  170  ██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████
       >1024:    8  ████████


## 4. First 10 chunks (with section path)

Each chunk shows its **heading hierarchy** (`section_path`) so you can see the table-of-contents context.

In [7]:
SHOW_N = 10

for i, ch in enumerate(chunks[:SHOW_N]):
    path_str = " > ".join(ch.section_path) if ch.section_path else "(no heading)"
    text_preview = ch.text[:200].replace("\n", " ↵ ")
    if len(ch.text) > 200:
        text_preview += " …"

    print(f"{'═' * 72}")
    print(f"Chunk {i}  │  {ch.token_count} tokens  │  pages {ch.page_start}–{ch.page_end}  │  type: {ch.block_type}")
    print(f"Section: {path_str}")
    print(f"Text   : {text_preview}")
    print()

════════════════════════════════════════════════════════════════════════
Chunk 0  │  463 tokens  │  pages 0–0  │  type: paragraph
Section: (no heading)
Text   : Mục lục ↵  ↵ GIỚI THIỆU .............................................................................5 ↵  ↵ Chương 1. MỞ ĐẦU ............................................................7 ↵  ↵ 1.1. KHÁI NIỆM CƠ BẢ …

════════════════════════════════════════════════════════════════════════
Chunk 1  │  305 tokens  │  pages 0–1  │  type: paragraph
Section: 2.4.3. Biểu thức điều kiện trong các cấu trúc điều khiển 43
Text   : 2.4.3. Biểu thức điều kiện trong các cấu trúc điều khiển 43 ↵  ↵ Chương 3. LỚP VÀ ĐỐI TƯỢNG .................................... 48 ↵  ↵ 3.1. TẠO VÀ SỬ DỤNG ĐỐI TƯỢNG ............................ 49 ↵  ↵ 3.2. TƯƠ …

════════════════════════════════════════════════════════════════════════
Chunk 2  │  96 tokens  │  pages 1–1  │  type: paragraph
Section: 5.1. PHƯƠNG THỨC VÀ TRẠNG THÁI ĐỐI TƯỢNG70
Text   : 5.1. PHƯƠ

## 5. Inspect a specific chunk

Change `CHUNK_IDX` to drill into any chunk.

In [8]:
CHUNK_IDX = 150  # ← change this

ch = chunks[CHUNK_IDX]

print(f"chunk_id     : {ch.chunk_id}")
print(f"doc_id       : {ch.doc_id}")
print(f"block_type   : {ch.block_type}")
print(f"token_count  : {ch.token_count}")
print(f"pages        : {ch.page_start} – {ch.page_end}")
print(f"chapter      : {ch.chapter}")
print(f"section      : {ch.section}")
print(f"section_path : {ch.section_path}")
print(f"block_ids    : {ch.block_ids}")
print(f"\n{'─' * 72}")
print(ch.text)

chunk_id     : bb0030519171
doc_id       : java-demo
block_type   : paragraph
token_count  : 1015
pages        : 109 – 110
chapter      : Chương 13. LẬP TRÌNH TỔNG QUÁT VÀ CÁC LỚP COLLECTION
section      : 7.6. LỢI ÍCH CỦA QUAN HỆ THỪA KẾ
section_path : ['Chương 13. LẬP TRÌNH TỔNG QUÁT VÀ CÁC LỚP COLLECTION', '7.6. LỢI ÍCH CỦA QUAN HỆ THỪA KẾ']
block_ids    : ['408953fe3868', 'c5479f5d306a', 'ab12d10fb959', '2fef19ccb9bb', '0ba4509a0c3c', 'a1cc46f30d04']

────────────────────────────────────────────────────────────────────────
7.6. LỢI ÍCH CỦA QUAN HỆ THỪA KẾ

Quan hệ thừa kế trong thiết kế mang lại cho ta rất nhiều điều.

Lợi ích thứ nhất: tránh lặp các đoạn mã bị trùng lặp. Ta có thể loại bỏ được 
những đoạn mã trùng lặp bằng cách tách ra các hành vi chung của một nhóm các lớp 
đối tượng và đưa phần mã đó vào một lớp cha. Nhờ đó, khi ta cần sửa nó, ta chỉ cần 
cập nhật mã ở duy nhất một nơi, và sửa đổi đó có hiệu lực tại tất cả các lớp kế thừa hành 
vi đó. Công việc gói gọn trong việ